# 0. Libraries

In [ ]:
pip install simplemma

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import warnings

# Filter out the specific UserWarnings
warnings.filterwarnings("ignore", category=UserWarning, message="A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy")
warnings.filterwarnings("ignore", category=UserWarning, message="unable to load libtensorflow_io_plugins.so")
warnings.filterwarnings("ignore", category=UserWarning, message="file system plugins are not loaded")

In [3]:
# Accuracy metrics from Scikit-Learn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

In [4]:
# Hugging Face library
from datasets import Dataset, DatasetDict

In [5]:
# NLP libraries
import re
import nltk
import simplemma

from simplemma import text_lemmatizer
from nltk.corpus import stopwords

# 1. Load Dataset

In [6]:
def load_data(file_path):
    return pd.read_csv(file_path, header=None, delimiter='\t', names = ['emotion','text'])

train_path = '/kaggle/input/emotion/train-emotion-all.tsv'
test_path = '/kaggle/input/emotion/test-emotion-all.tsv'
val_path = '/kaggle/input/emotion/valid-emotion-all.tsv'

df_train = load_data(train_path)
df_test = load_data(test_path)
df_val = load_data(val_path)

In [7]:
# To get an idea of the data
pd.set_option('display.max_colwidth', 150)
df_train.head()

,emotion,text
0,NEUTRA,Il ministro della Salute
1,GIOIA,". “Ho parlato di #bodyshaming e #fatshaming in Parlamento, perché credo che solo nominarli faccia bene e perchè molta pubblicistica ha relegato qu..."
2,NEUTRA,Se gli editori dell ' Aie sono in polemica per l ' intervento sugli sconti
3,RABBIA,"1 Maggio 2020 #Milano, duplice violenza sessuale: preso maniaco seriale nigeriano Clandestinamente in Italia dal 2018 aveva segnalazioni per atti ..."
4,AMORE,amore mio per sempre


In [8]:
# Remove user mention here. could not do it in the preprocess function
df_train['text'] = df_train['text'].str.replace('@[A-Za-z0-9]+\s?', '', regex=True)
df_val['text'] = df_val['text'].str.replace('@[A-Za-z0-9]+\s?', '', regex=True)
df_test['text'] = df_test['text'].str.replace('@[A-Za-z0-9]+\s?', '', regex=True)

In [9]:
# I'm combining the pandas dataframe to the dataset dictionary of Hugging Face

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)
val_dataset = Dataset.from_pandas(df_val)

# Create the DatasetDict
dataset = DatasetDict({'train': train_dataset, 'test': test_dataset, 'validation': val_dataset})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['emotion', 'text'],
        num_rows: 1854
    })
    test: Dataset({
        features: ['emotion', 'text'],
        num_rows: 546
    })
    validation: Dataset({
        features: ['emotion', 'text'],
        num_rows: 327
    })
})


In [10]:
# Removing duplicates

# Initialize a dictionary to store updated datasets
updated_datasets = {}

# Check for and remove duplicates in each split
for split in dataset.keys():
    split_data = dataset[split]
    
    # Access the 'text' column within the list
    text_column = split_data['text']
    
    # Initialize a set to track unique texts
    unique_texts = set()
    
    # Initialize lists to store the filtered data
    filtered_text = []
    
    # Iterate through the 'text' column and filter duplicates
    for text in text_column:
        if text not in unique_texts:
            unique_texts.add(text)
            filtered_text.append(text)
    
    # Create a new Dataset object with the filtered data
    updated_datasets[split] = split_data.select(list(range(len(filtered_text))))
    
    # Print the number of removed duplicates
    duplicate_count = len(text_column) - len(filtered_text)
    print(f"Duplicates removed in {split} split: {duplicate_count}\n")

# Update the dataset dictionary with the filtered datasets
dataset.update(updated_datasets)

# Print the updated dataset information
for split in dataset.keys():
    split_data = dataset[split]
    print(f"{split}: {len(split_data['text'])} rows")

print(dataset)

Duplicates removed in train split: 73

Duplicates removed in test split: 3

Duplicates removed in validation split: 9

train: 1781 rows
test: 543 rows
validation: 318 rows
DatasetDict({
    train: Dataset({
        features: ['emotion', 'text'],
        num_rows: 1781
    })
    test: Dataset({
        features: ['emotion', 'text'],
        num_rows: 543
    })
    validation: Dataset({
        features: ['emotion', 'text'],
        num_rows: 318
    })
})


# 2. Data Prepocessing

In [11]:
italian_stopwords = set(stopwords.words('italian'))

# Define a function to preprocess text
def preprocess_text(text):    
    # Tokenization, lemmatization, removing punctuation, stopwords and URLs
    text = text_lemmatizer(text, lang='it')
    text = ' '.join(text)
    
    text = re.sub(r'[^\w\s\']', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    text = ' '.join(word for word in text.split() if word.lower() not in italian_stopwords)
    
    return text




def preprocess_dataset(dataset):
    dataset['text'] = preprocess_text(dataset['text'])
    return dataset

dataset = dataset.map(preprocess_dataset)

  0%|          | 0/1781 [00:00<?, ?ex/s]

  0%|          | 0/543 [00:00<?, ?ex/s]

  0%|          | 0/318 [00:00<?, ?ex/s]

In [12]:
dataset['train']['text'][0:5]

['ministro Salute',
 'avere parlare bodyshaming fatshaming parlamento credere solo nominarli bene perchè molto pubblicistica avere relegare tema qualcosa riguardare donna potere avere significare parlare voce maschile',
 "editore ' aia essere polemica ' intervento sconto",
 '1 maggio 2020 Milano duplice violenza sessuale prendere maniaco seriale nigeriano Clandestinamente Italia 2018 avere segnalazione atto persecutore violenza sessuale resistenza pubblico ufficiale finalmente finire dietro sbarra sinistro volere ginocchio',
 'amore sempre']

In [13]:
# Convert the dataset to be ready for vectorization
X_train = np.array(dataset['train']['text'])
Y_train = np.array(dataset['train']['emotion'])

X_val = np.array(dataset['validation']['text'])
Y_val = np.array(dataset['validation']['emotion'])

X_test = np.array(dataset['test']['text'])
Y_test = np.array(dataset['test']['emotion'])


# 3. Feature Extraction

## 3.1 Bag of Words 

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
x_train = cv.fit_transform(X_train)
x_test = cv.transform(X_test)

## 3.2 TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
x_train = tv.fit_transform(X_train)
x_test = tv.transform(X_test)

# 4. Support Vector Machine

In [15]:
# LinearSVC
from sklearn.svm import SVC
svm = SVC(random_state=0)

In [16]:
svm.fit(x_train,Y_train)

y_test_svm=svm.predict(x_test)

# 5. Naive Bayes

In [17]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()

In [18]:
nb.fit(x_train,Y_train)

y_test_nb=nb.predict(x_test)

# 6. Metrics

In [19]:
report_svm = classification_report(Y_test, y_test_svm)

report_nb = classification_report(Y_test, y_test_nb)

print("Support Vector Machine Classification Report:")
print(report_svm)

print("\nNaive Bayes Classification Report:")
print(report_nb)

Support Vector Machine Classification Report:
              precision    recall  f1-score   support

       AMORE       0.64      0.43      0.51        21
       GIOIA       0.92      0.37      0.53        59
      NEUTRA       0.48      0.97      0.64       149
       PAURA       0.98      0.63      0.77        79
      RABBIA       0.83      0.59      0.69       102
    SORPRESA       0.94      0.37      0.53        46
   TRISTEZZA       0.80      0.59      0.68        87

    accuracy                           0.65       543
   macro avg       0.80      0.56      0.62       543
weighted avg       0.76      0.65      0.65       543


Naive Bayes Classification Report:
              precision    recall  f1-score   support

       AMORE       0.64      0.33      0.44        21
       GIOIA       0.66      0.56      0.61        59
      NEUTRA       0.79      0.83      0.81       149
       PAURA       0.81      0.70      0.75        79
      RABBIA       0.61      0.85      0.71       

In [20]:
accuracy_svm = accuracy_score(Y_test, y_test_svm) # (TP+TN)/P+N i.e total number of corrected classified tweet over total number of tweets

accuracy_nb = accuracy_score(Y_test, y_test_nb)

print("Support Vector Machine accuracy:", accuracy_svm)
print("Naive Bayes accuracy:", accuracy_nb)

Support Vector Machine accuracy: 0.6500920810313076
Naive Bayes accuracy: 0.7090239410681399


In [ ]:
precision_svm = precision_score(Y_test, y_test_svm,average=None, labels=['TRISTEZZA','GIOIA','AMORE','RABBIA','PAURA','SORPRESA','NEUTRA']) # TP/(TP+FP) i.e if predicted a certain class, which is the probability of being really that class?

precision_nb = precision_score(Y_test, y_test_nb,average=None, labels=['TRISTEZZA','GIOIA','AMORE','RABBIA','PAURA','SORPRESA','NEUTRA'])

print("Support Vector Machine precision:", precision_svm)
print("Naive Bayes precision:", precision_nb)

In [ ]:
recall_svm = recall_score(Y_test, y_test_svm,average=None, labels=['TRISTEZZA','GIOIA','AMORE','RABBIA','PAURA','SORPRESA','NEUTRA']) # TP/(TP+FN) i.e the ability of the estimator to predict all the tweets of a given class

recall_nb = recall_score(Y_test, y_test_nb,average=None, labels=['TRISTEZZA','GIOIA','AMORE','RABBIA','PAURA','SORPRESA','NEUTRA'])


print("Support Vector Machine recall:", recall_svm)
print("Naive Bayes recall:", recall_nb)

In [ ]:
f1score_svm = f1_score(Y_test, y_test_svm,average=None, labels=['TRISTEZZA','GIOIA','AMORE','RABBIA','PAURA','SORPRESA','NEUTRA']) # 2*(precision*recall)/(precision+recall)

f1score_nb = f1_score(Y_test, y_test_nb,average=None, labels=['TRISTEZZA','GIOIA','AMORE','RABBIA','PAURA','SORPRESA','NEUTRA'])


print("Support Vector Machine f1-score:", f1score_svm)
print("Naive Bayes f1-score:", f1score_nb)